# Lab 19: Does Machine Learning Find Hidden Alpha?

*Financial Engineering and the Evidence*, Chapter 19.

**Claim on trial:** machine-learning models find return predictability that linear models miss, and it can be turned into profitable portfolios.

You will train five models on a panel of 149 portfolios, test them out of sample from 1975, and check whether any edge survives large portfolios, recent years, trading costs and a change of random seed. Before each step, write down a prediction.

`FAST = True` uses one seed and refits every 10 years (a few minutes); `False` matches the full run (about 10 minutes).

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab19
from fintest.ml import build_panel, make_models

SOURCE = "french"     # "french" downloads from the Kenneth French library; "sim" uses fake data to test the code
FILES = {}            # optional: {"ff3": ..., "sbm100": ..., "ind49": ...} paths to saved zip files
GW_FILE = None        # optional: Goyal-Welch spreadsheet for Part D
FAST = True
TEST_START = "1975-01"
REFIT = 10 if FAST else 5
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab19.load(SOURCE, FILES)
SIM = d["simulated"]
panel = build_panel(d["ret"], d["mkt"], d.get("size"), d.get("nfirms"), d.get("bm"))
models = make_models(fast=FAST)
print(panel["asset"].nunique(), "portfolios,", panel.index.nunique(), "months | simulated:", SIM, "|", d.get("notes"))

## Part A: The horse race

**Predict first:** will gradient-boosted trees or the neural network beat OLS on size, value and momentum? By how much, in R² and in long–short Sharpe ratio?

In [ ]:
a = lab19.part_a(panel, models, TEST_START, REFIT)
lab19.fig_r2(a, OUT, SIM, TEST_START)
lab19.fig_cumulative(a, OUT, SIM)
display(Image(OUT / "fig19_1_r2.png"))
display(Image(OUT / "fig19_2_cumulative.png"))
a["table"].round(3)

## Part B: Robustness

**Predict first:** what happens to the best model's edge among large portfolios, after 2000 and at 25 bp per unit of turnover?

In [ ]:
b = lab19.part_b(a["res"]["pred"], list(models), d["large"])
lab19.fig_robust(b, OUT, SIM)
display(Image(OUT / "fig19_3_robustness.png"))
display(b["large"].round(3))
display(b["periods"].round(3))
b["costs"].round(2)

## Part C: Seeds and importance

**Predict first:** how much will the neural network's results change if only the random seed changes? Which characteristics will matter most?

In [ ]:
c = lab19.part_c(panel, FAST, TEST_START, 10, a["res"]["params"])
lab19.fig_importance(c, OUT, SIM)
display(Image(OUT / "fig19_4_importance.png"))
display(c["seeds"].round(3))
(100 * c["importance"]).round(4)

## Part D: Complexity in market timing

**Predict first:** with 12 months of training data, does a model with 1,200 random features forecast the market better than one with 2?

In [ ]:
dd = lab19.part_d(SOURCE, GW_FILE)
if dd is not None:
    lab19.fig_complexity(dd, OUT, SIM)
    display(Image(OUT / "fig19_5_complexity.png"))
    display(dd.round(3))

## Your verdict

1. Do the nonlinear models beat the linear ones out of sample, and is the difference significant?
2. Does the edge survive large portfolios, recent years and trading costs?
3. Is it hidden alpha, or a better way of combining characteristics that were already known?

Write one paragraph, using the chapter's verdict scale: survives, weakens, fails or provisional.